In [ ]:
from datasets import load_dataset

ds = load_dataset("Gustavosta/Stable-Diffusion-Prompts")
ds.save_to_disk("prompts/stable_diffusion_prompts")
print("数据集已保存到 dataset/stable_diffusion_prompts")

In [ ]:
from datasets import load_dataset, load_from_disk

# 从本地磁盘加载已保存的数据集
dataset = load_from_disk("../prompts/stable_diffusion_prompts")

# 打印数据集的基本结构
print("=" * 50)
print("数据集结构信息:")
print("=" * 50)
print(f"数据集类型: {type(dataset)}")
print(f"数据集划分: {list(dataset.keys())}")

# 打印每个划分的详细信息
for split_name, split_data in dataset.items():
    print(f"/n{split_name}划分详情:")
    print(f"  样本数量: {len(split_data)}")
    print(f"  特征字段: {split_data.column_names}")
    print(f"  特征结构: {split_data.features}")

# 打印前3条样本
print("/n" + "=" * 50)
print("前3条样本内容:")
print("=" * 50)

for split_name, split_data in dataset.items():
    print(f"/n{split_name}划分的前3条样本:")
    for i in range(min(3, len(split_data))):  # 确保不超过实际样本数
        print(f"/n样本 {i + 1}:")
        for column in split_data.column_names:
            print(f"  {column}: {split_data[i][column]}")

数据集结构信息:
数据集类型: <class 'datasets.dataset_dict.DatasetDict'>
数据集划分: ['train', 'test']

train划分详情:
  样本数量: 73718
  特征字段: ['Prompt']
  特征结构: {'Prompt': Value('string')}

test划分详情:
  样本数量: 8192
  特征字段: ['Prompt']
  特征结构: {'Prompt': Value('string')}

前3条样本内容:

train划分的前3条样本:

样本 1:
  Prompt: realistic car 3 d render sci - fi car and sci - fi robotic factory structure in the coronation of napoleon painting and digital billboard with point cloud in the middle, unreal engine 5, keyshot, octane, artstation trending, ultra high detail, ultra realistic, cinematic, 8 k, 1 6 k, in style of zaha hadid, in style of nanospace michael menzelincev, in style of lee souder, in plastic, dark atmosphere, tilt shift, depth of field,

样本 2:
  Prompt: a comic potrait of a female necromamcer with big and cute eyes, fine - face, realistic shaded perfect face, fine details. night setting. very anime style. realistic shaded lighting poster by ilya kuvshinov katsuhiro, magali villeneuve, artgerm, jeremy lipkin and 

In [4]:
print(dataset['train'][0]['Prompt'])

realistic car 3 d render sci - fi car and sci - fi robotic factory structure in the coronation of napoleon painting and digital billboard with point cloud in the middle, unreal engine 5, keyshot, octane, artstation trending, ultra high detail, ultra realistic, cinematic, 8 k, 1 6 k, in style of zaha hadid, in style of nanospace michael menzelincev, in style of lee souder, in plastic, dark atmosphere, tilt shift, depth of field,


对比结果是否可复现

In [ ]:
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim

def compare_images(image_path1, image_path2):
    try:
        # 读取图片
        img1 = cv2.imread(image_path1)
        img2 = cv2.imread(image_path2)
        
        # 检查图片是否加载成功
        if img1 is None or img2 is None:
            raise FileNotFoundError("图片加载失败，请检查文件路径")
        
        # 检查图片尺寸是否一致
        if img1.shape != img2.shape:
            raise ValueError("图片尺寸不一致，无法比较差异")
        
        # 转换为灰度图
        gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
        
        # 计算结构相似性指数 (SSIM)
        (score, diff) = ssim(gray1, gray2, full=True)
        similarity_ssim = score  # 范围 [-1, 1]，值越大越相似
        
        # 计算欧几里得距离相似度
        dist = np.linalg.norm(gray1 - gray2)
        max_pixel = np.max(gray1)
        similarity_euclidean = 100 * (1 - dist / max_pixel)  # 转换为百分比
        
        # 输出结果
        print("===== 图片相似度对比 =====")
        print(f"SSIM相似度: {similarity_ssim:.4f} (范围 [-1, 1])")
        print(f"欧氏距离相似度: {similarity_euclidean:.2f}%")
        
        # 可视化差异图（可选）
        diff_normalized = cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX)
        diff_color = cv2.applyColorMap(diff_normalized.astype('uint8'), cv2.COLORMAP_HOT)
        cv2.imshow('差异热力图', diff_color)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
        
        return {
            'ssim': similarity_ssim,
            'euclidean_similarity': similarity_euclidean,
            'diff_image': diff_normalized
        }
    
    except Exception as e:
        print(f"错误: {str(e)}")
        return None

# 使用示例
if __name__ == "__main__":
    result = compare_images('/home/knight/watermark/Watermark/dataset/test/p1.png', '/home/knight/watermark/Watermark/dataset/origin/000000.png')

===== 图片相似度对比 =====
SSIM相似度: 1.0000 (范围 [-1, 1])
欧氏距离相似度: 100.00%
